# 🔧 Sprint 1 — Day 5: Tuning, Evaluation & Sprint Review

**BinX Tech • AI & Machine Learning Internship • Phase 3, Sprint 1 (Week 6)**

## 📖 Overview

This notebook closes Sprint 1 by systematically tuning the Keras neural network, using validation evidence to select a configuration, applying `EarlyStopping` and `ModelCheckpoint`, and assembling the evidence required for the Sprint Review.

The workflow follows the Day 5 curriculum: change one hyperparameter at a time, inspect validation loss, keep the best model, compare against the Day 1 baseline, and finish with a Sprint Review and Retrospective.

## 🎯 Learning Objectives

- Tune a neural network systematically, one variable at a time.
- Tune learning rate, network size, dropout, and batch size.
- Use EarlyStopping and ModelCheckpoint effectively.
- Assemble loss curves and a metric table for Sprint Review.
- Compare the final tuned network with the Day 1 baseline.
- Complete the Sprint 1 Review and Retrospective.

## 📊 Dataset

Same **Kaggle Credit Card Fraud Detection Dataset** used in the Phase 3 fraud-detection project. The target is `Class` (`0` = normal, `1` = fraud).

No synthetic results are used in this notebook. Dataset-dependent cells must be executed with the same project dataset used on Days 1–4.

In [ ]:
# 📦 Imports + reproducibility
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)
print("TensorFlow:", tf.__version__)

In [ ]:
# 📂 Load the project dataset: local file first, then Google Drive as a fallback
from pathlib import Path

DATA_PATH = Path("creditcard.csv")

if not DATA_PATH.exists():
    try:
        import gdown
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
    GDRIVE_FILE_ID = "1UZ3hAYkcXulu8MSdZQiSzpgb1qu4bXNh"
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", str(DATA_PATH), quiet=False)

for alt in ["/content/creditcard.csv", "/content/drive/MyDrive/creditcard.csv"]:
    if not DATA_PATH.exists() and os.path.exists(alt):
        DATA_PATH = Path(alt)

if not DATA_PATH.exists():
    raise FileNotFoundError("creditcard.csv was not found. Upload/mount the same project dataset used on Days 1-4.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 🔎 Data-quality + leakage checks
assert "Class" in df.columns
assert df.isna().sum().sum() == 0
num = df.select_dtypes(include=np.number)
assert np.isfinite(num.to_numpy()).all()
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows (before removal):", int(df.duplicated().sum()))

# 🧹 Remove exact duplicates, matching the Day 1 baseline's data preparation
# (this keeps the Day 1 vs. Day 5 comparison on an identical, fair basis)
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
removed = before - len(df)
print(f"Exact duplicate rows removed: {removed:,}")
print(f"Rows after duplicate removal: {len(df):,}")
assert df.duplicated().sum() == 0

print("\nTarget distribution (post-dedup):")
display(df["Class"].value_counts().sort_index())


In [ ]:
# ✂️ Stratified train / validation / test split
X = df.drop(columns="Class")
y = df["Class"].astype("int32")
X_train, X_tmp, y_train, y_tmp = train_test_split(X,y,test_size=0.30,stratify=y,random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=RANDOM_STATE)
print(X_train.shape, X_val.shape, X_test.shape)
assert not set(X_train.index)&set(X_val.index)
assert not set(X_train.index)&set(X_test.index)
assert not set(X_val.index)&set(X_test.index)

In [ ]:
# 📏 Fit preprocessing on training data only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
assert np.isfinite(X_train_s).all() and np.isfinite(X_val_s).all() and np.isfinite(X_test_s).all()

## 🆚 Day 1 Baseline

The curriculum requires every later model to be compared with the baseline. If the exact Day 1 baseline metric is already stored in the project, replace the placeholder variables below with that recorded result. This notebook does **not** invent a baseline score.

In [ ]:
# 🧱 Recompute a transparent Logistic Regression baseline on the same split
baseline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
baseline.fit(X_train,y_train)
p = baseline.predict_proba(X_test)[:,1]
baseline_metrics = {"Accuracy":accuracy_score(y_test,p>=0.5),"ROC-AUC":roc_auc_score(y_test,p),"PR-AUC":average_precision_score(y_test,p)}
baseline_metrics

# 🖥️ Hands-On Lab: Sprint 1 Close-Out

1. Tune one hyperparameter at a time and record validation performance.
2. Add EarlyStopping and confirm training halts while retaining the best weights.
3. Assemble baseline vs. neural-network evidence, architecture, and loss curves.
4. Complete the GitHub/mentor review workflow outside the notebook.
5. Present the Sprint Review and write the Retrospective.

In [ ]:
# 🧠 Model factory
def build_model(lr=0.001, units1=64, units2=32, dropout=0.30):
    model=Sequential([Input(shape=(X_train_s.shape[1],)),Dense(units1,activation="relu"),BatchNormalization(),Dropout(dropout),Dense(units2,activation="relu"),Dense(1,activation="sigmoid")])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),loss="binary_crossentropy",metrics=["accuracy"])
    return model

def run_trial(name, lr=0.001, units1=64, units2=32, dropout=0.30, batch=32, epochs=30):
    tf.keras.backend.clear_session(); tf.random.set_seed(RANDOM_STATE)
    m=build_model(lr,units1,units2,dropout)
    h=m.fit(X_train_s,y_train,validation_data=(X_val_s,y_val),epochs=epochs,batch_size=batch,verbose=0)
    return {"Experiment":name,"Learning Rate":lr,"Network":f"{units1}-{units2}","Dropout":dropout,"Batch Size":batch,"Best Val Loss":min(h.history["val_loss"]),"Best Val Accuracy":max(h.history["val_accuracy"]),"Epochs":len(h.history["loss"])}

In [ ]:
# 🎚️ One-variable-at-a-time tuning
trials=[]
for lr in [0.0001,0.001,0.01]: trials.append(run_trial(f"Learning rate {lr}",lr=lr))
for u1,u2 in [(32,16),(64,32),(128,64)]: trials.append(run_trial(f"Network {u1}-{u2}",units1=u1,units2=u2))
for d in [0.0,0.30,0.50]: trials.append(run_trial(f"Dropout {d}",dropout=d))
for b in [16,32,64]: trials.append(run_trial(f"Batch size {b}",batch=b))
experiment_log=pd.DataFrame(trials).sort_values("Best Val Loss").reset_index(drop=True)
display(experiment_log)

In [ ]:
# 📈 Compare validation-loss evidence
fig, ax=plt.subplots(figsize=(10,5))
# Re-run representative curves so the evidence is visible in one figure.
for lr in [0.0001,0.001,0.01]:
    tf.keras.backend.clear_session(); tf.random.set_seed(RANDOM_STATE)
    m=build_model(lr=lr); h=m.fit(X_train_s,y_train,validation_data=(X_val_s,y_val),epochs=30,batch_size=32,verbose=0)
    ax.plot(h.history["val_loss"],label=f"LR={lr}")
ax.set_title("Learning-rate tuning — validation loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss"); ax.legend(); ax.grid(alpha=.25); plt.show()

## 🏆 Selection Rule

The final configuration is selected using **validation loss**, not the test set. This preserves the test set as a final, held-out evaluation source.

In [ ]:
best=experiment_log.iloc[0]
print("Selected configuration:")
display(best.to_frame("Value"))
best_lr=float(best["Learning Rate"]); best_u1,best_u2=map(int,str(best["Network"]).split("-")); best_drop=float(best["Dropout"]); best_batch=int(best["Batch Size"])

In [ ]:
# ⏹️ EarlyStopping + ModelCheckpoint
tf.keras.backend.clear_session(); tf.random.set_seed(RANDOM_STATE)
final_model=build_model(best_lr,best_u1,best_u2,best_drop)
checkpoint_path="best_sprint1_model.keras"
es=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True,verbose=1)
ckpt=ModelCheckpoint(checkpoint_path,monitor="val_loss",mode="min",save_best_only=True,verbose=1)
final_history=final_model.fit(X_train_s,y_train,validation_data=(X_val_s,y_val),epochs=100,batch_size=best_batch,callbacks=[es,ckpt],verbose=1)
print("Epochs actually run:",len(final_history.history["loss"]))
print("Checkpoint exists:",os.path.exists(checkpoint_path))

In [ ]:
# 📉 Final training curves
fig,ax=plt.subplots(figsize=(10,5)); ax.plot(final_history.history["loss"],label="Training loss"); ax.plot(final_history.history["val_loss"],label="Validation loss"); ax.set_title("Final model — loss curves"); ax.set_xlabel("Epoch"); ax.set_ylabel("Binary cross-entropy"); ax.legend(); ax.grid(alpha=.25); plt.show()

In [ ]:
# 📊 Final evaluation + baseline comparison
test_loss,test_acc=final_model.evaluate(X_test_s,y_test,verbose=0)
prob=final_model.predict(X_test_s,verbose=0).ravel()
final_metrics={"Accuracy":test_acc,"ROC-AUC":roc_auc_score(y_test,prob),"PR-AUC":average_precision_score(y_test,prob),"Test Loss":test_loss}
comparison=pd.DataFrame({"Metric":list(final_metrics.keys()),"Day 1 Baseline":[baseline_metrics.get(k,np.nan) for k in final_metrics],"Final Tuned NN":list(final_metrics.values())})
display(comparison)

In [ ]:
# 🏗️ Final architecture
final_model.summary()

# 👨‍🏫 Sprint Review Evidence

The Sprint Review should demonstrate: baseline score, trained neural-network architecture and score, tuning experiments, loss curves, and the final metric table. The curriculum explicitly requires these artifacts before the model is demoed.

In [ ]:
# 🔬 Objective → Evidence mapping
objective_evidence=pd.DataFrame({"Objective":["Systematic tuning","EarlyStopping","Best model checkpoint","Baseline comparison","Sprint evidence"],"Evidence":["experiment_log","final_history","best_sprint1_model.keras","comparison","loss curves + architecture + metric table"]})
display(objective_evidence)

# 🔄 Sprint 1 Retrospective

### What went well
The sprint connected neural-network fundamentals to a practical Keras workflow and then to systematic tuning and evaluation.

### What could be improved
As experiment count grows, manually recording configurations becomes harder to maintain and compare.

### 🎯 Concrete change for Sprint 2
Log every experiment's configuration and results in a structured experiment-tracking workflow from the start.

# 📌 Acceptance Criteria

- Notebook runs without errors.
- Tuning experiments are documented.
- EarlyStopping is used.
- ModelCheckpoint preserves the best model.
- Loss curves and metric table are included.
- Results are compared with the baseline.
- Results are documented in Markdown.
- GitHub branch/PR and mentor approval are verified outside notebook execution.

These criteria follow the Week 6 Sprint 1 requirements.

# 💭 Reflection

The main lesson from Day 5 is that tuning is a controlled experiment, not random parameter changes. Changing one variable at a time makes the validation evidence easier to interpret, while EarlyStopping and ModelCheckpoint make the training process more disciplined.

Sprint 1 therefore closes with a complete chain: **baseline → neural network → training → tuning → evaluation → review → retrospective**.

# 🏁 Conclusion

Day 5 completes Sprint 1 by tuning and evaluating the neural network, preserving the best model, assembling the required evidence, and documenting the Sprint Review and Retrospective.